<a href="https://colab.research.google.com/github/Skye-Zhangg/Homework/blob/main/hw5_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 3 (55 points): NLP and Attention Mechanism

Part 1 (10 points): Implement the scaled dot-product attention as discussed in class
(lecture 10) from scratch (use NumPy and pandas only, no deep learning libraries are
allowed for this step).

In [ ]:
import numpy as np
import pandas as pd

# softmax

def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)  # stability
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def attention(Q, K, V, mask=None):
    dk = Q.shape[-1]

    # compute attention scores
    scores = Q @ K.transpose(0, 1, 3, 2)
    scores = scores / np.sqrt(dk)

    # apply mask
    if mask is not None:
        scores = scores + (mask - 1) * 1e9

    # normalize
    weights = softmax(scores)

    # weighted sum
    output = weights @ V

    return output, weights


# -------- test --------
np.random.seed(0)

batch_size = 2
heads = 1
seq_len = 4
dim = 8

Q = np.random.rand(batch_size, heads, seq_len, dim)
K = np.random.rand(batch_size, heads, seq_len, dim)
V = np.random.rand(batch_size, heads, seq_len, dim)

mask = np.ones((batch_size, 1, seq_len, seq_len))

out, w = attention(Q, K, V, mask)

print("Output shape:", out.shape)
print("Attention shape:", w.shape)

print("Output:", pd.DataFrame(out[0, 0]))
print("\nAttention Weights:\n", pd.DataFrame(w[0, 0]))

Output shape: (2, 1, 4, 8)
Attention shape: (2, 1, 4, 4)
Output:           0         1         2         3         4         5         6  \
0  0.531654  0.557341  0.375107  0.653477  0.630209  0.542586  0.401301   
1  0.526577  0.568905  0.374014  0.655253  0.654644  0.545314  0.397046   
2  0.539267  0.548843  0.369644  0.657737  0.608549  0.548240  0.401048   
3  0.536953  0.563642  0.352191  0.673887  0.632643  0.568005  0.388801   

          7  
0  0.575864  
1  0.581497  
2  0.572260  
3  0.574374  

Attention Weights:
           0         1         2         3
0  0.265119  0.237333  0.216678  0.280871
1  0.232625  0.265621  0.215814  0.285940
2  0.290977  0.203406  0.230218  0.275399
3  0.252390  0.207520  0.253824  0.286266


Part 2 (10 points): Pick any encoder-decoder seq2seq model (as discussed in class) and
integrate the scaled dot-product attention in the encoder architecture. You may come
up with your own technique of integration or adopt one from literature. Hint: See
Bahdanau or Luong attention paper presented in class (lecture 10).

In [ ]:
import numpy as np
import pandas as pd

class SimpleSeq2Seq:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim

        # weights
        self.W_e = np.random.randn(input_dim, hidden_dim) * 0.01
        self.W_d = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.W_o = np.random.randn(hidden_dim, output_dim) * 0.01

    def attention(self, query, keys, values, mask=None):
        dk = query.shape[-1]

        scores = query @ keys.transpose(0, 2, 1)
        scores = scores / np.sqrt(dk)

        if mask is not None:
            scores += (mask[:, None, :] - 1) * 1e9   # 改写mask逻辑

        weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
        weights /= np.sum(weights, axis=-1, keepdims=True)

        context = weights @ values
        return context, weights

    def encode(self, x):
        return np.tanh(x @ self.W_e)

    def decode(self, enc_out, target_len, mask=None):
        batch_size = enc_out.shape[0]
        h = np.zeros((batch_size, self.hidden_dim))

        outputs = []
        attn_list = []

        for _ in range(target_len):
            q = h[:, None, :]

            context, weights = self.attention(q, enc_out, enc_out, mask)
            attn_list.append(weights)

            # update hidden state
            h = np.tanh(context[:, 0, :] @ self.W_d + h)

            out = h @ self.W_o
            outputs.append(out)

        outputs = np.stack(outputs, axis=1)
        attn_list = np.stack(attn_list, axis=1)

        return outputs, attn_list

    def forward(self, x, target, mask=None):
        enc_out = self.encode(x)
        outputs, attn = self.decode(enc_out, target.shape[1], mask)
        return outputs, attn


# -------- test --------
np.random.seed(0)

batch_size = 2
input_dim = 5
hidden_dim = 8
output_dim = 10
seq_len = 6
target_len = 4

inputs = np.random.randn(batch_size, seq_len, input_dim)
target = np.random.randn(batch_size, target_len, output_dim)
mask = np.ones((batch_size, seq_len))

model = SimpleSeq2Seq(input_dim, hidden_dim, output_dim)

outputs, attention_weights = model.forward(inputs, target, mask)

# output
print("\nOutputs:\n", pd.DataFrame(outputs[0]))
print("\nAttention Weights:\n", pd.DataFrame(attention_weights[0, 0]))


Outputs:
           0         1         2         3         4         5         6  \
0  0.000002  0.000011  0.000002  0.000014  0.000011 -0.000001 -0.000002   
1  0.000005  0.000022  0.000005  0.000029  0.000022 -0.000003 -0.000004   
2  0.000007  0.000033  0.000007  0.000043  0.000032 -0.000004 -0.000006   
3  0.000009  0.000044  0.000010  0.000057  0.000043 -0.000005 -0.000007   

          7         8         9  
0 -0.000003  0.000014 -0.000005  
1 -0.000005  0.000028 -0.000009  
2 -0.000008  0.000041 -0.000014  
3 -0.000011  0.000055 -0.000019  

Attention Weights:
           0         1         2         3         4         5
0  0.166667  0.166667  0.166667  0.166667  0.166667  0.166667


Part 3 (5 points): Pick any public dataset of your choice (use a small-scale dataset like a
subset of the Tatoeba or Multi30k dataset) for machine translation task. Train your
model from Part 2 for the machine translation task. Evaluate test set by reporting the
BLEU Score.

In [ ]:

!pip3 install torch torchtext spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.2 MB/s eta 0:00:00


In [ ]:

!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:

!python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 68.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


# Data

pairs = [
    ("i am going home", "ich gehe nach hause"),
    ("she is reading a book", "sie liest ein buch"),
    ("he plays football", "er spielt fußball"),
    ("we like to travel", "wir reisen gerne"),
    ("they are cooking dinner", "sie kochen abendessen")
]

# Build vocab

def build_vocab(sentences):
    vocab = {"<pad>":0, "<sos>":1, "<eos>":2}
    tokenized = []

    for s in sentences:
        tokens = s.split()
        tokenized.append(tokens)
        for w in tokens:
            if w not in vocab:
                vocab[w] = len(vocab)

    return tokenized, vocab

en_sent = [p[0] for p in pairs]
de_sent = [p[1] for p in pairs]

tok_en, vocab_en = build_vocab(en_sent)
tok_de, vocab_de = build_vocab(de_sent)

def encode(sentences, vocab):
    return [[vocab["<sos>"]] + [vocab[w] for w in s] + [vocab["<eos>"]] for s in sentences]

en_idx = encode(tok_en, vocab_en)
de_idx = encode(tok_de, vocab_de)

# Model

class SeqModel(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.emb_enc = nn.Embedding(in_dim, 32)
        self.emb_dec = nn.Embedding(out_dim, 32)

        self.encoder = nn.LSTM(32, 64)
        self.decoder = nn.LSTM(32, 64)

        self.fc = nn.Linear(64, out_dim)

    def encode(self, src):
        emb = self.emb_enc(src)
        _, (h, c) = self.encoder(emb)
        return h, c

    def decode(self, trg, h, c):
        emb = self.emb_dec(trg)
        out, (h, c) = self.decoder(emb, (h, c))
        pred = self.fc(out.squeeze(0))
        return pred, h, c

# Init

model = SeqModel(len(vocab_en), len(vocab_de))
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Train
def run_epoch():
    model.train()
    total = 0

    for src, trg in zip(en_idx, de_idx):
        src = torch.LongTensor(src).unsqueeze(1)
        trg = torch.LongTensor(trg)

        optimizer.zero_grad()

        h, c = model.encode(src)

        loss = 0
        inp = trg[0]

        for t in range(1, len(trg)):
            out, h, c = model.decode(inp.view(1,1), h, c)
            loss += criterion(out, trg[t].unsqueeze(0))
            inp = trg[t]

        loss.backward()
        optimizer.step()

        total += loss.item()

    return total / len(en_idx)

# BLEU
def compute_bleu():
    model.eval()
    idx2word = {v:k for k,v in vocab_de.items()}

    scores = []

    for src, ref in zip(en_idx, tok_de):
        src = torch.LongTensor(src).unsqueeze(1)

        with torch.no_grad():
            h, c = model.encode(src)

        inp = torch.LongTensor([vocab_de["<sos>"]])
        pred_words = []

        for _ in range(15):
            with torch.no_grad():
                out, h, c = model.decode(inp.view(1,1), h, c)

            token = out.argmax(1).item()
            if token == vocab_de["<eos>"]:
                break

            pred_words.append(idx2word[token])
            inp = torch.LongTensor([token])

        smooth = SmoothingFunction().method1
        score = sentence_bleu([ref], pred_words, smoothing_function=smooth)
        scores.append(score)

    return sum(scores) / len(scores)


# Train loop
for epoch in range(10):
    loss = run_epoch()
    print(f"Epoch: {epoch+1}, Train Loss: {loss:.3f}")

bleu = compute_bleu()
print(f"BLEU Score: {bleu:.3f}")

Epoch: 1, Train Loss: 13.011
Epoch: 2, Train Loss: 12.725
Epoch: 3, Train Loss: 12.472
Epoch: 4, Train Loss: 12.202
Epoch: 5, Train Loss: 11.896
Epoch: 6, Train Loss: 11.537
Epoch: 7, Train Loss: 11.102
Epoch: 8, Train Loss: 10.570
Epoch: 9, Train Loss: 9.939
Epoch: 10, Train Loss: 9.243
BLEU Score: 0.016


Part 4 (30 points): In this part you are required to implement a simplified Transformer
model from scratch (using Python and NumPy/PyTorch/TensorFlow with minimal highlevel abstractions) and apply it to a machine translation task (e.g., English-to-French or
English-to-German translation) using the same dataset from part 3.
We discussed Transformer architecture in depth in class (Vaswani Paper – Attention is
all you need). Apply the following simplifications to the original model architecture:
1. Reduced Model Depth: Use 2 encoder layers and 2 decoder layers instead of
the standard 6.
2. Limited Attention Heads: Use 2 attention heads in the multi-head attention
mechanism rather than 8.
3. Smaller Embedding Size: Set the embedding dimension to 64 instead of 512.
4. Reduced Feedforward Network Size: Use a feedforward dimension of 128
instead of 2048.
5. Smaller Dataset: Use a small dataset (e.g., about 10k sentence pairs).
6. Tokenization Simplifications: Use a basic subword tokenizer (like Byte Pair
Encoding - BPE) or word-level tokenization instead of complex language-specific tokenizers.

Key components to implement:

1. Positional Encoding: Implement Sinusoidal position encoding.
2. Scaled dot-product attention: Use the same implementation from part 1.
3. Multi-Head Attention: Integrate the scaled dot-product attention into a multi- head attention framework using the specified simplifications.
4. Encoder and Decoder Blocks: Implement simplified encoder and decoder layers, ensuring: Layer normalization, Residual connections, Masked attention in the decoder for autoregressive generation.
5. Final Output Layer: Implement a linear layer followed by a SoftMax activation for generating translated tokens.

Evaluation: Compute the BLEU score on a validation set and compare the performance with your model from part 2. Explain why there are differences in performance. Also discuss any other differences you notice, for example runtime etc.

The attention weights in the Seq2Seq model (Part 2) are nearly uniform, indicating that the model fails to learn meaningful alignment between input and output tokens. This suggests that the attention mechanism is not effectively focusing on relevant parts of the input sequence, which leads to poor translation performance.

In contrast, the Transformer model (Part 4) achieves a higher BLEU score and produces more accurate translations. By using multi-head self-attention, the Transformer is able to capture global dependencies between words and model relationships across the entire sequence simultaneously, rather than relying on sequential processing as in RNN-based models.

Furthermore, the Transformer benefits from parallel computation and a more flexible architecture, allowing it to learn richer representations of the data. Therefore, the Transformer clearly outperforms the Seq2Seq model in both attention quality and overall translation performance, even when trained on a very small dataset.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

device = torch.device("cpu")

# -----------------------------
# Dataset
# -----------------------------
pairs = [
    ("i am going home", "ich gehe nach hause"),
    ("she is reading a book", "sie liest ein buch"),
    ("he plays football", "er spielt fußball"),
    ("we like to travel", "wir reisen gerne"),
    ("they are cooking dinner", "sie kochen abendessen")
]

# -----------------------------
# Build vocab
# -----------------------------
def build_vocab(sentences):
    vocab = {"<pad>":0, "<sos>":1, "<eos>":2}
    tokenized = []
    for s in sentences:
        tokens = s.split()
        tokenized.append(tokens)
        for w in tokens:
            if w not in vocab:
                vocab[w] = len(vocab)
    return tokenized, vocab

en_sent = [p[0] for p in pairs]
de_sent = [p[1] for p in pairs]

tok_en, vocab_en = build_vocab(en_sent)
tok_de, vocab_de = build_vocab(de_sent)

def encode(sentences, vocab):
    return [[vocab["<sos>"]] + [vocab[w] for w in s] + [vocab["<eos>"]] for s in sentences]

en_idx = encode(tok_en, vocab_en)
de_idx = encode(tok_de, vocab_de)

vocab_de_inv = {v:k for k,v in vocab_de.items()}

# -----------------------------
# Positional Encoding
# -----------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        for pos in range(max_len):
            for i in range(0, d_model, 2):
                pe[pos, i] = math.sin(pos / (10000 ** (i/d_model)))
                if i+1 < d_model:
                    pe[pos, i+1] = math.cos(pos / (10000 ** (i/d_model)))
        self.pe = pe.unsqueeze(1)

    def forward(self, x):
        return x + self.pe[:x.size(0)]

# -----------------------------
# Multi-head Attention
# -----------------------------
class MultiHead(nn.Module):
    def __init__(self, d_model=64, heads=2):
        super().__init__()
        self.heads = heads
        self.d_k = d_model // heads

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.fc = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V):
        seq_len_q, batch, _ = Q.shape
        seq_len_k, _, _ = K.shape

        Q = self.q(Q)
        K = self.k(K)
        V = self.v(V)

        Q = Q.view(seq_len_q, batch, self.heads, self.d_k).transpose(1,2)
        K = K.view(seq_len_k, batch, self.heads, self.d_k).transpose(1,2)
        V = V.view(seq_len_k, batch, self.heads, self.d_k).transpose(1,2)

        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)

        out = out.transpose(1,2).contiguous().view(seq_len_q, batch, -1)
        return self.fc(out)

# -----------------------------
# Feed Forward
# -----------------------------
class FFN(nn.Module):
    def __init__(self, d_model=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,128),
            nn.ReLU(),
            nn.Linear(128,d_model)
        )
    def forward(self,x):
        return self.net(x)

# -----------------------------
# Encoder Layer
# -----------------------------
class EncoderLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = MultiHead()
        self.ffn = FFN()
        self.norm1 = nn.LayerNorm(64)
        self.norm2 = nn.LayerNorm(64)

    def forward(self,x):
        x = self.norm1(x + self.attn(x, x, x))
        x = self.norm2(x + self.ffn(x))
        return x

# -----------------------------
# Decoder Layer
# -----------------------------
class DecoderLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.self_attn = MultiHead()
        self.enc_attn = MultiHead()
        self.ffn = FFN()

        self.norm1 = nn.LayerNorm(64)
        self.norm2 = nn.LayerNorm(64)
        self.norm3 = nn.LayerNorm(64)

    def forward(self, x, enc):
        # self-attention
        x = self.norm1(x + self.self_attn(x, x, x))

        # cross-attention
        x = self.norm2(x + self.enc_attn(x, enc, enc))

        # feedforward
        x = self.norm3(x + self.ffn(x))

        return x

# -----------------------------
# Transformer
# -----------------------------
class Transformer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.emb_enc = nn.Embedding(input_dim,64)
        self.emb_dec = nn.Embedding(output_dim,64)
        self.pos = PositionalEncoding(64)

        self.enc_layers = nn.ModuleList([EncoderLayer() for _ in range(2)])
        self.dec_layers = nn.ModuleList([DecoderLayer() for _ in range(2)])

        self.fc = nn.Linear(64,output_dim)

    def forward(self,src,trg):
        src = self.pos(self.emb_enc(src))
        trg = self.pos(self.emb_dec(trg))

        for layer in self.enc_layers:
            src = layer(src)

        for layer in self.dec_layers:
            trg = layer(trg,src)

        return self.fc(trg)

# -----------------------------
# Training
# -----------------------------
model = Transformer(len(vocab_en),len(vocab_de))
optimizer = optim.Adam(model.parameters(),lr=0.001)
criterion = nn.CrossEntropyLoss()

def train():
    model.train()
    total_loss=0

    for src,trg in zip(en_idx,de_idx):
        src = torch.LongTensor(src).unsqueeze(1)
        trg = torch.LongTensor(trg).unsqueeze(1)

        optimizer.zero_grad()

        output = model(src,trg[:-1])
        output_dim = output.shape[-1]

        loss = criterion(
            output.view(-1,output_dim),
            trg[1:].reshape(-1)
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss/len(en_idx)

# -----------------------------
# BLEU
# -----------------------------
def evaluate():
    model.eval()
    smooth = SmoothingFunction().method1
    scores=[]

    for src,trg in zip(en_idx,tok_de):
        src = torch.LongTensor(src).unsqueeze(1)

        pred=[vocab_de["<sos>"]]

        for _ in range(15):
            trg_tensor = torch.LongTensor(pred).unsqueeze(1)
            with torch.no_grad():
                out = model(src,trg_tensor)

            next_token = out.argmax(-1)[-1].item()
            if next_token==vocab_de["<eos>"]:
                break
            pred.append(next_token)

        pred_words=[vocab_de_inv[i] for i in pred[1:]]
        score = sentence_bleu([trg],pred_words,smoothing_function=smooth)
        scores.append(score)

    return sum(scores)/len(scores)

# -----------------------------
# Train loop
# -----------------------------
for epoch in range(10):
    loss = train()
    print(f"Epoch: {epoch+1}, Train Loss: {loss:.3f}")

bleu = evaluate()
print(f"BLEU Score: {bleu:.3f}")

# -----------------------------
# Translation example
# -----------------------------
def translate(sentence):
    tokens = sentence.split()
    src = [vocab_en["<sos>"]] + [vocab_en[w] for w in tokens] + [vocab_en["<eos>"]]
    src = torch.LongTensor(src).unsqueeze(1)

    pred=[vocab_de["<sos>"]]

    for _ in range(15):
        trg = torch.LongTensor(pred).unsqueeze(1)
        with torch.no_grad():
            out = model(src,trg)
        token = out.argmax(-1)[-1].item()
        if token==vocab_de["<eos>"]:
            break
        pred.append(token)

    return " ".join([vocab_de_inv[i] for i in pred[1:]])

print("\nSample Translations:")
for en,de in zip(en_sent,de_sent):
    print("EN:",en)
    print("GT:",de)
    print("PR:",translate(en))
    print("-"*30)

Epoch: 1, Train Loss: 3.201
Epoch: 2, Train Loss: 2.452
Epoch: 3, Train Loss: 2.056
Epoch: 4, Train Loss: 1.628
Epoch: 5, Train Loss: 1.231
Epoch: 6, Train Loss: 0.914
Epoch: 7, Train Loss: 0.664
Epoch: 8, Train Loss: 0.482
Epoch: 9, Train Loss: 0.356
Epoch: 10, Train Loss: 0.274
BLEU Score: 0.625

Sample Translations:
EN: i am going home
GT: ich gehe nach hause
PR: ich gehe nach hause
------------------------------
EN: she is reading a book
GT: sie liest ein buch
PR: sie liest ein buch
------------------------------
EN: he plays football
GT: er spielt fußball
PR: 
------------------------------
EN: we like to travel
GT: wir reisen gerne
PR: wir reisen gerne
------------------------------
EN: they are cooking dinner
GT: sie kochen abendessen
PR: sie kochen abendessen
------------------------------
